# **Prepare images for the Cellpose command line (batch processing)**

This notebook loops over **one or more folders** of images, pulls out the channels Cellpose needs, and saves them as small **TIFF files**. It then writes the exact **Cellpose command-line (CLI) commands** needed to segment all of them in one go, using the pretrained `cyto3` model (cells) and `nuclei` model (nuclei).

The notebook does **not** run Cellpose itself, so it is quick. The segmentation is run afterwards from a terminal (see the last step).

### What you can choose
| Option | Meaning |
|---|---|
| Cell segmentation input | a **composite** of several channels (e.g. the cytoplasmic organelle channels) or a **single channel** (e.g. the plasma membrane) |
| Add nuclei to the cell segmentation | give `cyto3` the nuclear channel as a helper (a 2-channel image) or not (a 1-channel image) |
| Segment nuclei | run the `nuclei` model on the nuclear channel |

### Can the nuclei and the cells share one TIFF file?
**Yes, when the nuclei are used as a helper channel for the cells.** The prepared file then has two channels (channel 1 = cell signal, channel 2 = nuclei). `cyto3` uses both, and the `nuclei` model is simply told to read channel 2 only (`--chan 2 --chan2 0`). I tested this: it gives the same result as a separate nuclei-only file, and the same file is not stored twice.

**No, when the cells are segmented from the cell signal alone.** The cell file then has a single channel, so a separate nuclei-only file is saved for the `nuclei` model.

The notebook picks the right layout for you and tells you which one it used.

### The six steps
1. Settings (folders, channels, Cellpose parameters)
2. Find the images
3. Prepare the TIFF files (one per image)
4. Write the Cellpose commands
5. Run the commands in a terminal
6. Use the results

### What is created
Inside `OUTPUT_ROOT`, one sub-folder per source folder:

| Folder | Content |
|---|---|
| `cell_input\` | TIFF files for the cell segmentation (and nuclei too if the file is shared) |
| `nuclei_input\` | nuclei-only TIFF files (only when a separate file is needed) |
| `metadata\` | one `.json` file per image: original file, pixel size, downsampling, channel use |
| `cell_masks\`, `nuclei_masks\` | empty now; Cellpose writes the masks here |

Plus `run_cellpose_batch.bat`, which holds the commands.

---------------------
## **STEP 1: Settings**

#### &#x1F6D1; &#x270D; **User Input Required:**

| Setting | Meaning |
|---|---|
| `SOURCE_FOLDERS` | One or more folders containing your images. Add more entries to the list to process several folders. **All images in all listed folders are prepared and segmented with the same settings below** (only the Z/XY pixel-size ratio is read from each image). If different sample types need different settings (channels, diameters, thresholds, ...), run this notebook once per sample type, each with its own `SOURCE_FOLDERS` and `OUTPUT_ROOT`. |
| `IM_TYPE` | File type of the images (e.g. `".czi"`, `".tiff"`). |
| `OUTPUT_ROOT` | Folder where the prepared files and the command file are saved. |
| `SEGMENT_CELLS`, `SEGMENT_NUCLEI` | Which segmentations to prepare. |
| `CELL_SOURCE` | `"composite"` = cell signal is a weighted average of channels (set `COMPOSITE_WEIGHTS`); `"single"` = the one channel `PM_CH`. |
| `PM_CH` | Plasma membrane channel number (used when `CELL_SOURCE = "single"`, and for the optional inversion). |
| `COMPOSITE_WEIGHTS` | For a composite: `{channel number: weight}`. A larger weight makes that channel count more; channels not listed (or with weight 0) are left out. |
| `INVERT_PM` | Composite only: also invert the plasma membrane channel (so the *inside* of the cell is bright). Only has an effect if `PM_CH` has a weight above 0 in `COMPOSITE_WEIGHTS`. |
| `NUC_CH` | Nuclear stain channel number. |
| `USE_NUCLEI_FOR_CELLS` | `True` = give the cell model the nuclei as a helper channel (2-channel file, shared with the nuclei model). `False` = cell signal only. |
| `XY_DOWNSAMPLE` | Shrinks images in XY by this factor (2 = half size) to speed up Cellpose. Keep 1 for no shrinking. The masks Cellpose returns will then be at the **shrunken** size; the factor is stored in the metadata. |
| `CELL_DIAMETER`, `NUC_DIAMETER` | Typical object width in XY, in pixels **of the prepared (possibly shrunken) image**. Use 0 to let Cellpose estimate it. |
| `FLOW_THRESHOLD`, `CELLPROB_THRESHOLD` | Cellpose sensitivity. Lower `CELLPROB_THRESHOLD` (e.g. -3) finds more/larger objects. Higher `FLOW_THRESHOLD` accepts more irregular shapes. |
| `MIN_SIZE_CELL`, `MIN_SIZE_NUC` | Smallest object kept (in voxels of the prepared image). |
| `BATCH_SIZE` | Image tiles processed together. Lower it if the GPU runs out of memory. |
| `USE_GPU` | `True` to use the GPU in the command line run. |
| `SKIP_EXISTING` | `True` = do not redo images that were already prepared (lets you re-run the notebook after adding images). |

Channel numbers start at 0, as in the other notebooks. The values below are starting points, not tuned for every sample.

In [ ]:
#### USER INPUT REQUIRED ####
SOURCE_FOLDERS = [
    Path(r"C:\Users\Shannon\Desktop\Cell-nuc-mask_test-data\example-images"),
    # Path(r"C:\path\to\another\folder"),
]
IM_TYPE = ".tiff"
OUTPUT_ROOT = Path(r"C:\Users\Shannon\Desktop\Cell-nuc-mask_test-data\cellpose_batch")

SEGMENT_CELLS = True
SEGMENT_NUCLEI = True

# --- cell signal ---
CELL_SOURCE = "composite"      # "composite" or "single"
PM_CH = 7                      # plasma membrane channel
COMPOSITE_WEIGHTS = {0: 1, 
                     1: 2, 
                     2: 3, 
                     3: 3, 
                     4: 2, 
                     5: 2, 
                     6: 3}     # channel: weight (used if CELL_SOURCE = "composite")
INVERT_PM = False              # composite only: also add the inverted plasma membrane channel

# --- nuclei ---
NUC_CH = 1                     # nuclear stain channel
USE_NUCLEI_FOR_CELLS = True    # True: 2-channel cell file (cells + nuclei, shared with the nuclei model)

# --- image size ---
XY_DOWNSAMPLE = 1              # >1 shrinks XY (faster); masks are then at the shrunken size

# --- Cellpose parameters (diameters in pixels of the prepared image; 0 = estimate automatically) ---
CELL_DIAMETER = 500
NUC_DIAMETER = 150
FLOW_THRESHOLD = 0.2
CELLPROB_THRESHOLD = -3
MIN_SIZE_CELL = 5000
MIN_SIZE_NUC = 2000
BATCH_SIZE = 4
USE_GPU = True

SKIP_EXISTING = True

#### &#x1F3C3; **Run code; no user input required**

Loads the tools used below and checks that your settings make sense. (If you see an error here, read the message: it says which setting to change.)

In [ ]:
import json
import numpy as np
import pandas as pd
import tifffile
from pathlib import Path
from skimage.transform import resize

from infer_subc.core.file_io import read_czi_image, list_image_files
from infer_subc.organelles.membrane import membrane_composite

if not (SEGMENT_CELLS or SEGMENT_NUCLEI):
    raise ValueError("Nothing to do: set SEGMENT_CELLS and/or SEGMENT_NUCLEI to True.")
if CELL_SOURCE not in ("composite", "single"):
    raise ValueError('CELL_SOURCE must be "composite" or "single".')
if CELL_SOURCE == "composite" and not any(w > 0 for w in COMPOSITE_WEIGHTS.values()):
    raise ValueError("COMPOSITE_WEIGHTS needs at least one channel with a weight above 0.")
if not isinstance(SOURCE_FOLDERS, (list, tuple)):
    raise ValueError("SOURCE_FOLDERS must be a list, e.g. [Path(r'C:\\data')].")
if int(XY_DOWNSAMPLE) < 1:
    raise ValueError("XY_DOWNSAMPLE must be 1 or larger.")
XY_DOWNSAMPLE = int(XY_DOWNSAMPLE)

# Decide which files are needed (the nuclei channel is part of the cell file only when it helps the cells)
CELL_FILE_HAS_NUC = bool(SEGMENT_CELLS and USE_NUCLEI_FOR_CELLS)
NEED_NUC_CHANNEL = bool(CELL_FILE_HAS_NUC or SEGMENT_NUCLEI)
SEPARATE_NUC_FILE = bool(SEGMENT_NUCLEI and not CELL_FILE_HAS_NUC)

if SEGMENT_CELLS and CELL_FILE_HAS_NUC:
    layout = "ONE 2-channel file per image (channel 1 = cell signal, channel 2 = nuclei)"
    if SEGMENT_NUCLEI:
        layout += ", shared by the cell and nuclei models"
elif SEGMENT_CELLS:
    layout = "cell file (1 channel)" + (" + a separate nuclei-only file" if SEGMENT_NUCLEI else "")
else:
    layout = "nuclei-only file (1 channel)"
print("File layout:", layout)

needed_channels = [NUC_CH] if NEED_NUC_CHANNEL else []
if SEGMENT_CELLS:
    needed_channels += [c for c, w in COMPOSITE_WEIGHTS.items() if w > 0] if CELL_SOURCE == "composite" else [PM_CH]
    if CELL_SOURCE == "composite" and INVERT_PM:
        needed_channels.append(PM_CH)
print("Channels used:", sorted(set(needed_channels)))

---------------------
## **STEP 2: Find the images**

#### &#x1F3C3; **Run code; no user input required**

Lists the images found in each source folder. Each folder gets its own sub-folder inside `OUTPUT_ROOT`; if two folders have the same name, a number is added so nothing is overwritten.

In [ ]:
datasets, used_names = [], set()
for folder in SOURCE_FOLDERS:
    folder = Path(folder)
    if not folder.exists():
        print(f"WARNING: folder not found, skipped: {folder}")
        continue
    files = list_image_files(folder, IM_TYPE)
    if not files:
        print(f"WARNING: no '{IM_TYPE}' files found in {folder}")
        continue
    name = base = folder.name or "images"
    n = 2
    while name in used_names:
        name, n = f"{base}_{n}", n + 1
    used_names.add(name)
    datasets.append({"name": name, "folder": folder, "files": files})

if not datasets:
    raise FileNotFoundError("No images found. Check SOURCE_FOLDERS and IM_TYPE in the settings.")

pd.set_option("display.max_colwidth", None)
pd.DataFrame([{"output sub-folder": d["name"], "source folder": str(d["folder"]), "images": len(d["files"])} for d in datasets])

---------------------
## **STEP 3: Prepare the TIFF files**

#### &#x1F3C3; **Run code; no user input required**

For every image: read it, build the cell signal (composite or single channel) and/or the nuclei channel, scale each to the range 0-1 (1st to 99.8th percentile), shrink in XY if requested, and save as a TIFF. A small `.json` file with the pixel size and settings is saved next to it.

This can take a little while for large images. If one image fails, the error is recorded and the rest continue.

In [ ]:
def _norm01(v):
    v = np.asarray(v, dtype=np.float32)
    p1, p99 = np.percentile(v, (1, 99.8))
    return np.clip((v - p1) / (p99 - p1 + 1e-8), 0, 1).astype(np.float32)


def _shrink(vol):
    if XY_DOWNSAMPLE == 1:
        return vol
    shape = (vol.shape[0], round(vol.shape[1] / XY_DOWNSAMPLE), round(vol.shape[2] / XY_DOWNSAMPLE))
    return resize(vol, shape, order=1, anti_aliasing=True).astype(np.float32)


def _cell_signal(img):
    """Cell signal as ZYX, from a composite of channels or a single channel."""
    if CELL_SOURCE == "single":
        return _norm01(img[PM_CH])
    weights = {f"weight_ch{c}": w for c, w in COMPOSITE_WEIGHTS.items()}
    comp = membrane_composite(img, Invert_PM=INVERT_PM, PM_Channel=PM_CH, **weights)
    return _norm01(comp)


records = []
for ds in datasets:
    root = Path(OUTPUT_ROOT) / ds["name"]
    dirs = {k: root / k for k in ("cell_input", "nuclei_input", "metadata", "cell_masks", "nuclei_masks")}
    ds["dirs"] = dirs
    for k, d in dirs.items():
        if (k == "cell_input" and SEGMENT_CELLS) or (k == "nuclei_input" and SEPARATE_NUC_FILE) \
           or (k == "cell_masks" and SEGMENT_CELLS) or (k == "nuclei_masks" and SEGMENT_NUCLEI) or k == "metadata":
            d.mkdir(parents=True, exist_ok=True)

    for src in ds["files"]:
        stem = Path(src).stem
        cell_f = dirs["cell_input"] / f"{stem}.tif"
        nuc_f = dirs["nuclei_input"] / f"{stem}.tif"
        meta_f = dirs["metadata"] / f"{stem}.json"
        expected = [meta_f] + ([cell_f] if SEGMENT_CELLS else []) + ([nuc_f] if SEPARATE_NUC_FILE else [])
        rec = {"dataset": ds["name"], "image": stem, "status": "", "anisotropy": np.nan, "prepared shape (Z,Y,X)": "", "message": ""}
        try:
            if SKIP_EXISTING and all(f.exists() for f in expected):
                info = json.loads(meta_f.read_text())
                rec.update(status="skipped (already prepared)", anisotropy=info["anisotropy"],
                           **{"prepared shape (Z,Y,X)": str(tuple(info["prepared_shape_zyx"]))})
            else:
                img, meta = read_czi_image(src)
                bad = [c for c in set(needed_channels) if c >= img.shape[0]]
                if bad:
                    raise ValueError(f"image has {img.shape[0]} channels (0-{img.shape[0] - 1}) but channel(s) {bad} are needed")
                scale = [float(s) for s in meta["scale"]]
                aniso = scale[0] / (scale[1] * XY_DOWNSAMPLE)

                nuc = _shrink(_norm01(img[NUC_CH])) if NEED_NUC_CHANNEL else None
                cell = _shrink(_cell_signal(img)) if SEGMENT_CELLS else None
                if SEGMENT_CELLS:
                    out = np.stack([cell, nuc], axis=1) if CELL_FILE_HAS_NUC else cell    # ZCYX or ZYX
                    tifffile.imwrite(cell_f, out.astype(np.float32))
                if SEPARATE_NUC_FILE:
                    tifffile.imwrite(nuc_f, nuc.astype(np.float32))
                shape = (cell if cell is not None else nuc).shape
                meta_f.write_text(json.dumps({
                    "source_file": str(src), "scale_zyx": scale, "xy_downsample": XY_DOWNSAMPLE,
                    "anisotropy": aniso, "original_shape_zyx": list(img.shape[1:]), "prepared_shape_zyx": list(shape),
                    "cell_file_channels": ["cell signal", "nuclei"] if CELL_FILE_HAS_NUC else ["cell signal"],
                    "cell_source": CELL_SOURCE if SEGMENT_CELLS else None, "nuclei_channel": NUC_CH if NEED_NUC_CHANNEL else None,
                    "pm_channel": PM_CH, "composite_weights": COMPOSITE_WEIGHTS if CELL_SOURCE == "composite" else None,
                    "invert_pm": INVERT_PM}, indent=1))
                rec.update(status="prepared", anisotropy=round(aniso, 4), **{"prepared shape (Z,Y,X)": str(tuple(shape))})
                del img, nuc, cell
        except Exception as e:
            rec.update(status="ERROR", message=f"{type(e).__name__}: {e}")
        records.append(rec)
        print(f"{ds['name']} / {stem}: {rec['status']} {rec['message']}")

prep_table = pd.DataFrame(records)
prep_table.to_csv(Path(OUTPUT_ROOT) / "prepared_images.csv", index=False)
if (prep_table["status"] == "ERROR").any():
    print("\nSome images failed (see the 'message' column below). The others were prepared normally.")
prep_table

---------------------
## **STEP 4: Write the Cellpose commands**

#### &#x1F3C3; **Run code; no user input required**

Builds the Cellpose command(s) for every folder and saves them in `run_cellpose_batch.bat` inside `OUTPUT_ROOT`.

* The cell command runs `cyto3` (`--chan 1 --chan2 2` when the file has nuclei, `--chan 0 --chan2 0` for a single channel).
* The nuclei command runs `nuclei`. With a shared file it reads channel 2 only (`--chan 2 --chan2 0`).
* The pixel-size ratio between Z and XY (`--anisotropy`) comes from each image's metadata. If images in a folder have different values, a separate command is written per image.

In [ ]:
def _cli(indir_or_file, savedir, model, chan, chan2, diameter, min_size, aniso, channel_axis=None, is_file=False):
    parts = ["python -m cellpose",
             f'--image_path "{indir_or_file}"' if is_file else f'--dir "{indir_or_file}"',
             f"--pretrained_model {model}", "--do_3D", "--z_axis 0"]
    if channel_axis is not None:
        parts.append(f"--channel_axis {channel_axis}")
    parts += [f"--chan {chan}", f"--chan2 {chan2}", f"--diameter {diameter}", f"--anisotropy {aniso:.4f}",
              f"--flow_threshold {FLOW_THRESHOLD}", f"--cellprob_threshold {CELLPROB_THRESHOLD}",
              f"--min_size {min_size}", f"--batch_size {BATCH_SIZE}"]
    if USE_GPU:
        parts.append("--use_gpu")
    parts += ["--save_tif", "--no_npy", f'--savedir "{savedir}"', "--verbose"]
    return " ".join(parts)


good = prep_table[prep_table["status"].isin(["prepared", "skipped (already prepared)"])]
if good.empty:
    raise RuntimeError("No images were prepared successfully, so there are no commands to write. See the table above.")

lines = ["@echo off", "chcp 65001 >nul",
         "REM Run this file from a terminal where the environment with Cellpose 3 (cellpose<4) is activated.",
         "REM The masks are saved as <image name>_cp_masks.tif in the cell_masks / nuclei_masks folders.", ""]
for ds in datasets:
    sub = good[good["dataset"] == ds["name"]]
    if sub.empty:
        continue
    dirs = ds["dirs"]
    groups = sub.groupby(sub["anisotropy"].round(3))
    lines.append(f"REM ===== {ds['name']} =====")
    for targets in (("cells", SEGMENT_CELLS), ("nuclei", SEGMENT_NUCLEI)):
        name, wanted = targets
        if not wanted:
            continue
        if name == "cells":
            indir, savedir, model = dirs["cell_input"], dirs["cell_masks"], "cyto3"
            chan, chan2, axis = (1, 2, 1) if CELL_FILE_HAS_NUC else (0, 0, None)
            diam, msize = CELL_DIAMETER, MIN_SIZE_CELL
        else:
            savedir, model, diam, msize = dirs["nuclei_masks"], "nuclei", NUC_DIAMETER, MIN_SIZE_NUC
            if SEPARATE_NUC_FILE:
                indir, chan, chan2, axis = dirs["nuclei_input"], 0, 0, None
            else:
                indir, chan, chan2, axis = dirs["cell_input"], 2, 0, 1          # shared 2-channel file: nuclei = channel 2
        if len(groups) == 1:
            aniso = float(sub["anisotropy"].iloc[0])
            lines.append(f"REM {name}")
            lines.append(_cli(indir, savedir, model, chan, chan2, diam, msize, aniso, axis))
        else:
            lines.append(f"REM {name}: images have different Z/XY pixel-size ratios, so one command per image")
            for _, r in sub.iterrows():
                lines.append(_cli(Path(indir) / f"{r['image']}.tif", savedir, model, chan, chan2, diam, msize,
                                  float(r["anisotropy"]), axis, is_file=True))
    lines.append("")
lines.append("echo Done.")

bat_file = Path(OUTPUT_ROOT) / "run_cellpose_batch.bat"
bat_file.write_text("\r\n".join(lines) + "\r\n", encoding="utf-8")
print(f"Commands saved to:\n{bat_file}\n")
print("\n".join(lines))

---------------------
## **STEP 5: Run Cellpose from a terminal**

1. Open **Anaconda Prompt** (or any terminal).
2. Activate an environment that has **Cellpose 3** (`cellpose<4`; Cellpose 4 has no `cyto3`/`nuclei` models). For example: `conda activate waveorder-cytoland`.
3. Run the file printed above, e.g.
   `"C:\...\cellpose_batch\run_cellpose_batch.bat"`
   (or copy single lines from the printed commands and run them one at a time).

Notes:
* The **first time** Cellpose runs it uses the model files it already has. If it cannot find `cyto3` or `nuclei`, it will try to download them.
* Progress is shown in the terminal. Re-running the file overwrites existing masks.
* Output folders must already exist; this notebook creates them, so do not delete them.

---------------------
## **STEP 6: Use the results**

* Masks are saved as `<image name>_cp_masks.tif` (labels: 0 = background, 1, 2, 3, ... = objects) in `cell_masks\` and `nuclei_masks\`.
* If `XY_DOWNSAMPLE` was above 1, the masks are smaller than the original image. The `metadata\<image name>.json` file stores the factor and the original shape so they can be scaled back up.
* `prepared_images.csv` in `OUTPUT_ROOT` lists every image with its status.
* To match each nucleus to a cell, or to open masks next to the image in Napari, use the same approach as in `NEW_infer_masks_cellpose.ipynb`.

#### Troubleshooting
| Problem | What to do |
|---|---|
| "No images found" | Check `SOURCE_FOLDERS` (raw strings: `Path(r"C:\...")`) and `IM_TYPE`. |
| "image has N channels but channel(s) [..] are needed" | A channel number in the settings does not exist in that image. |
| `FileExistsError: --savedir ... does not exist` | A `cell_masks`/`nuclei_masks` folder was deleted. Re-run Step 3. |
| Cellpose reports it cannot find `cyto3` or `nuclei` | You are in a Cellpose 4 environment. Use one with `cellpose<4`. |
| Out of GPU memory | Lower `BATCH_SIZE` or raise `XY_DOWNSAMPLE`, then re-run Steps 3-4 with `SKIP_EXISTING = False`. |
| Changed a setting that affects the images (channels, weights, downsampling) | Set `SKIP_EXISTING = False` so the files are rebuilt. |